In [1]:
# ============================================================
# 14_AURORA_transaction_cost_and_rebalance_sensitivity.ipynb
#
# Robustness notebook for AURORA-TWETF paper
#
# Purpose:
# 1. Load final source-aware Notebook 13B return matrix.
# 2. Load AURORA/ROMA weights where available.
# 3. Reconstruct sensitivity returns under alternative:
#       - transaction costs: 0, 10, 25, 50 bps
#       - rebalance frequencies: daily, weekly, monthly, quarterly
# 4. Compare AURORA10-UAMV-B, ROMA-P4, ROMA-P2, ROMA-B12,
#    ROMA-B6, ROMA-B3, ROMA-B1, and selected AURORA benchmarks.
# 5. Produce paper-ready robustness tables and figures.
#
# Important interpretation:
# - This notebook is a robustness/sensitivity analysis.
# - It should not be used to select a new "best" method on the test set.
# - The primary method remains AURORA10-UAMV-B.
# - The primary claim remains downside-risk and risk-adjusted robustness,
#   not total-return dominance.
#
# Educational/research use only.
# Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

# ============================================================
# 1. Paths and run configuration
# ============================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

PROJECT_CODE = "ROMA_AURORA_TWETF"
AURORA_CODE = "AURORA_TWETF"
ROMA_CODE = "ROMA_TWETF"

# Final source-aware synthesis run from Notebook 13B.
NOTEBOOK13B_RUN_ID = "20260625_065916"

NOTEBOOK13B_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / PROJECT_CODE
    / "source_aware_unified_paper_comparison"
    / f"run_{NOTEBOOK13B_RUN_ID}"
)

NOTEBOOK13B_MATRIX_PATH = (
    NOTEBOOK13B_ROOT
    / "returns"
    / "notebook13B_source_aware_strict_test_return_matrix.parquet"
)

NOTEBOOK13B_OUTPUT_INDEX_PATH = (
    NOTEBOOK13B_ROOT
    / "tables"
    / "notebook13B_output_index.csv"
)

# AURORA Notebook 10 allocation run.
AURORA_NOTEBOOK10_RUN_ID = "20260624_100748"
AURORA_N10_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / AURORA_CODE
    / "uncertainty_aware_mean_variance_allocation"
    / f"run_{AURORA_NOTEBOOK10_RUN_ID}"
)

# ROMA R2 allocation run.
ROMA_R2_RUN_ID = "20260625_025314"
ROMA_R2_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / ROMA_CODE
    / "aligned_allocation_backtest"
    / f"run_{ROMA_R2_RUN_ID}"
)

ETF_RETURN_PANEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "panels"
    / "AURORA_etf_return_panel.parquet"
)

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / PROJECT_CODE
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = (
    OUTPUT_ROOT
    / "transaction_cost_rebalance_sensitivity"
    / f"run_{RUN_ID}"
)

TABLE_RUN_DIR = RUN_ROOT / "tables"
RETURN_RUN_DIR = RUN_ROOT / "returns"
FIGURE_RUN_DIR = RUN_ROOT / "figures"
PAPER_FIGURE_DIR = RUN_ROOT / "paper_figures"
REPORT_RUN_DIR = RUN_ROOT / "reports"
MANUSCRIPT_RUN_DIR = RUN_ROOT / "manuscript_assets"

GLOBAL_TABLE_DIR = OUTPUT_ROOT / "tables"
GLOBAL_REPORT_DIR = OUTPUT_ROOT / "reports"
GLOBAL_FIGURE_DIR = OUTPUT_ROOT / "figures"
GLOBAL_MANUSCRIPT_DIR = OUTPUT_ROOT / "manuscript_assets"

for d in [
    RUN_ROOT,
    TABLE_RUN_DIR,
    RETURN_RUN_DIR,
    FIGURE_RUN_DIR,
    PAPER_FIGURE_DIR,
    REPORT_RUN_DIR,
    MANUSCRIPT_RUN_DIR,
    GLOBAL_TABLE_DIR,
    GLOBAL_REPORT_DIR,
    GLOBAL_FIGURE_DIR,
    GLOBAL_MANUSCRIPT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 90)
print("Notebook 14: Transaction Cost and Rebalance Sensitivity")
print("=" * 90)
print("Timestamp UTC              :", RUN_TIMESTAMP)
print("Run ID                     :", RUN_ID)
print("Notebook 13B root          :", NOTEBOOK13B_ROOT)
print("Notebook 13B matrix        :", NOTEBOOK13B_MATRIX_PATH)
print("AURORA N10 root            :", AURORA_N10_ROOT)
print("ROMA R2 root               :", ROMA_R2_ROOT)
print("ETF return panel           :", ETF_RETURN_PANEL_PATH)
print("Run root                   :", RUN_ROOT)
print("=" * 90)

if not NOTEBOOK13B_MATRIX_PATH.exists():
    raise FileNotFoundError(f"Missing Notebook 13B matrix: {NOTEBOOK13B_MATRIX_PATH}")

# ============================================================
# 2. Global settings
# ============================================================

ANNUALIZATION = 252

PRIMARY_AURORA_POLICY = "AURORA10_UAMV_B_more60_defensive"
AURORA_SENSITIVITY_POLICIES = [
    "AURORA10_UAMV_D_low_turnover",
    "AURORA10_UAMV_E_no_regime_tilt_control",
    "AURORA10_UAMV_A_balanced",
    "AURORA10_UAMV_C_more60_growth",
    "AURORA10_validation_selected_UAMV",
]

PRIMARY_ROMA_POLICY = "ROMA_P4_balanced_regime_blend"
ROMA_SENSITIVITY_POLICIES = [
    "ROMA_P2_20d_only_return_seeking",
    "ROMA_P0_validation_selected_20_60_blend",
    "ROMA_P1_more20_return_seeking",
    "ROMA_P3_conservative_blend",
]

SOURCE_AWARE_BENCHMARKS = [
    "ROMA_B12_ma_timing_equal_weight",
    "ROMA_B6_00881_only",
    "ROMA_B3_0050_only",
    "ROMA_B1_equal_weight_all_etfs",
    "ROMA_B10_momentum_top2_63d",
    "ROMA_B15_minimum_variance_126d",
    "AURORA_B6_00881_only",
    "AURORA_B3_0050_only",
    "AURORA_B1_equal_weight_all_etfs",
]

PRIMARY_POLICIES_FOR_ROBUSTNESS = [
    PRIMARY_AURORA_POLICY,
    "AURORA10_UAMV_D_low_turnover",
    "AURORA10_UAMV_E_no_regime_tilt_control",
    PRIMARY_ROMA_POLICY,
    "ROMA_P2_20d_only_return_seeking",
    "ROMA_B12_ma_timing_equal_weight",
    "ROMA_B6_00881_only",
    "ROMA_B3_0050_only",
    "ROMA_B1_equal_weight_all_etfs",
    "AURORA_B6_00881_only",
    "AURORA_B1_equal_weight_all_etfs",
]

DISPLAY_NAMES = {
    "AURORA10_UAMV_B_more60_defensive": "AURORA10-UAMV-B",
    "AURORA10_UAMV_D_low_turnover": "AURORA10-UAMV-D",
    "AURORA10_UAMV_E_no_regime_tilt_control": "AURORA10-UAMV-E",
    "AURORA10_UAMV_A_balanced": "AURORA10-UAMV-A",
    "AURORA10_UAMV_C_more60_growth": "AURORA10-UAMV-C",
    "AURORA10_validation_selected_UAMV": "AURORA validation-selected",
    "ROMA_P4_balanced_regime_blend": "ROMA-P4",
    "ROMA_P2_20d_only_return_seeking": "ROMA-P2",
    "ROMA_P0_validation_selected_20_60_blend": "ROMA-P0",
    "ROMA_P1_more20_return_seeking": "ROMA-P1",
    "ROMA_P3_conservative_blend": "ROMA-P3",
    "ROMA_B12_ma_timing_equal_weight": "ROMA-B12 timing",
    "ROMA_B6_00881_only": "ROMA-B6 00881",
    "ROMA_B3_0050_only": "ROMA-B3 0050",
    "ROMA_B1_equal_weight_all_etfs": "ROMA-B1 equal-weight",
    "ROMA_B10_momentum_top2_63d": "ROMA-B10 momentum",
    "ROMA_B15_minimum_variance_126d": "ROMA-B15 min-var",
    "AURORA_B6_00881_only": "AURORA-B6 00881",
    "AURORA_B3_0050_only": "AURORA-B3 0050",
    "AURORA_B1_equal_weight_all_etfs": "AURORA-B1 equal-weight",
}

POLICY_GROUPS = {
    PRIMARY_AURORA_POLICY: "AURORA final",
    "AURORA10_UAMV_D_low_turnover": "AURORA sensitivity",
    "AURORA10_UAMV_E_no_regime_tilt_control": "AURORA sensitivity",
    "AURORA10_UAMV_A_balanced": "AURORA sensitivity",
    "AURORA10_UAMV_C_more60_growth": "AURORA sensitivity",
    "AURORA10_validation_selected_UAMV": "AURORA validation-selected",
    PRIMARY_ROMA_POLICY: "ROMA baseline",
    "ROMA_P2_20d_only_return_seeking": "ROMA baseline",
    "ROMA_P0_validation_selected_20_60_blend": "ROMA baseline",
    "ROMA_P1_more20_return_seeking": "ROMA baseline",
    "ROMA_P3_conservative_blend": "ROMA baseline",
    "ROMA_B12_ma_timing_equal_weight": "ROMA benchmark",
    "ROMA_B6_00881_only": "ROMA benchmark",
    "ROMA_B3_0050_only": "ROMA benchmark",
    "ROMA_B1_equal_weight_all_etfs": "ROMA benchmark",
    "ROMA_B10_momentum_top2_63d": "ROMA benchmark",
    "ROMA_B15_minimum_variance_126d": "ROMA benchmark",
    "AURORA_B6_00881_only": "AURORA benchmark",
    "AURORA_B3_0050_only": "AURORA benchmark",
    "AURORA_B1_equal_weight_all_etfs": "AURORA benchmark",
}

# Robustness grid.
TRANSACTION_COST_BPS_GRID = [0, 10, 25, 50]
REBALANCE_FREQUENCIES = {
    "daily": "D",
    "weekly": "W-FRI",
    "monthly": "M",
    "quarterly": "Q",
}

# Approximate annualization adjustment remains 252 because returns are daily.
# Rebalance frequency only changes strategy return path via weight holding.

# ============================================================
# 3. Utility functions
# ============================================================

def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path}")

def standardize_date_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"])
        out = out.set_index("date")
    else:
        out.index = pd.to_datetime(out.index)
    out = out.sort_index()
    return out

def write_json(path: Path, obj) -> None:
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def write_markdown(path: Path, text: str) -> None:
    Path(path).write_text(text, encoding="utf-8")

def save_table(df: pd.DataFrame, local_name: str, global_name: str | None = None):
    local_path = TABLE_RUN_DIR / local_name
    df.to_csv(local_path, index=False)
    if global_name is not None:
        global_path = GLOBAL_TABLE_DIR / global_name
        df.to_csv(global_path, index=False)
    else:
        global_path = None
    return local_path, global_path

def save_parquet_and_csv(df: pd.DataFrame, base_path: Path, index=True):
    csv_path = Path(str(base_path) + ".csv")
    pq_path = Path(str(base_path) + ".parquet")
    df.to_csv(csv_path, index=index)
    try:
        df.to_parquet(pq_path, index=index)
    except Exception as e:
        print("Parquet save skipped:", pq_path, repr(e))
        pq_path = None
    return pq_path, csv_path

def sha256_file(path: Path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root: Path) -> pd.DataFrame:
    root = Path(root)
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

def drawdown_from_equity(equity: pd.Series) -> pd.Series:
    equity = pd.Series(equity).astype(float)
    return equity / equity.cummax() - 1.0

def equity_from_returns(r: pd.Series) -> pd.Series:
    r = pd.Series(r).dropna().astype(float)
    return (1.0 + r).cumprod()

def performance_metrics(r: pd.Series, annualization=ANNUALIZATION) -> dict:
    r = pd.Series(r).dropna().astype(float)
    if len(r) == 0:
        return {
            "n_days": 0,
            "start_date": pd.NaT,
            "end_date": pd.NaT,
            "total_return": np.nan,
            "annual_return": np.nan,
            "annual_volatility": np.nan,
            "sharpe": np.nan,
            "sortino": np.nan,
            "max_drawdown": np.nan,
            "calmar": np.nan,
            "final_equity": np.nan,
            "mean_daily_return": np.nan,
            "daily_volatility": np.nan,
            "positive_day_rate": np.nan,
            "worst_daily_return": np.nan,
            "best_daily_return": np.nan,
        }

    r.index = pd.to_datetime(r.index)
    n = len(r)
    equity = equity_from_returns(r)
    total_return = float(equity.iloc[-1] - 1.0)
    annual_return = float(equity.iloc[-1] ** (annualization / n) - 1.0)

    daily_vol = float(r.std(ddof=1)) if n > 1 else np.nan
    annual_vol = float(daily_vol * np.sqrt(annualization)) if np.isfinite(daily_vol) else np.nan

    mean_daily = float(r.mean())
    sharpe = float((mean_daily / daily_vol) * np.sqrt(annualization)) if daily_vol and daily_vol > 0 else np.nan

    neg = r[r < 0]
    downside_vol = float(neg.std(ddof=1)) if len(neg) > 1 else np.nan
    sortino = float((mean_daily / downside_vol) * np.sqrt(annualization)) if np.isfinite(downside_vol) and downside_vol > 0 else np.nan

    dd = drawdown_from_equity(equity)
    max_dd = float(dd.min())
    calmar = float(annual_return / abs(max_dd)) if max_dd < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": r.index.min(),
        "end_date": r.index.max(),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_dd,
        "calmar": calmar,
        "final_equity": float(equity.iloc[-1]),
        "mean_daily_return": mean_daily,
        "daily_volatility": daily_vol,
        "positive_day_rate": float((r > 0).mean()),
        "worst_daily_return": float(r.min()),
        "best_daily_return": float(r.max()),
    }

def build_performance_table(return_matrix: pd.DataFrame, extra_cols: dict | None = None) -> pd.DataFrame:
    rows = []
    for col in return_matrix.columns:
        m = performance_metrics(return_matrix[col])
        m["strategy_name"] = col
        m["display_name"] = DISPLAY_NAMES.get(col, col)
        m["strategy_group"] = POLICY_GROUPS.get(col, "other")
        if extra_cols:
            m.update(extra_cols)
        rows.append(m)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out["rank_total_return"] = out["total_return"].rank(ascending=False, method="min")
    out["rank_annual_return"] = out["annual_return"].rank(ascending=False, method="min")
    out["rank_sharpe"] = out["sharpe"].rank(ascending=False, method="min")
    out["rank_sortino"] = out["sortino"].rank(ascending=False, method="min")
    out["rank_max_drawdown"] = out["max_drawdown"].rank(ascending=False, method="min")
    out["rank_calmar"] = out["calmar"].rank(ascending=False, method="min")
    out["composite_rank"] = (
        out["rank_total_return"]
        + out["rank_annual_return"]
        + out["rank_sharpe"]
        + out["rank_sortino"]
        + out["rank_max_drawdown"]
        + out["rank_calmar"]
    ) / 6.0

    out = out.sort_values(
        ["composite_rank", "sharpe", "total_return"],
        ascending=[True, False, False],
    ).reset_index(drop=True)

    return out

def paired_difference(strategy_returns: pd.Series, benchmark_returns: pd.Series) -> dict:
    s = pd.Series(strategy_returns).dropna().astype(float)
    b = pd.Series(benchmark_returns).dropna().astype(float)
    common = s.index.intersection(b.index).sort_values()
    s = s.loc[common]
    b = b.loc[common]

    ps = performance_metrics(s)
    pb = performance_metrics(b)
    excess = s - b

    return {
        "n_days": int(len(common)),
        "strategy_total_return": ps["total_return"],
        "benchmark_total_return": pb["total_return"],
        "diff_total_return": ps["total_return"] - pb["total_return"],
        "strategy_annual_return": ps["annual_return"],
        "benchmark_annual_return": pb["annual_return"],
        "diff_annual_return": ps["annual_return"] - pb["annual_return"],
        "strategy_sharpe": ps["sharpe"],
        "benchmark_sharpe": pb["sharpe"],
        "diff_sharpe": ps["sharpe"] - pb["sharpe"],
        "strategy_sortino": ps["sortino"],
        "benchmark_sortino": pb["sortino"],
        "diff_sortino": ps["sortino"] - pb["sortino"],
        "strategy_max_drawdown": ps["max_drawdown"],
        "benchmark_max_drawdown": pb["max_drawdown"],
        "drawdown_improvement": ps["max_drawdown"] - pb["max_drawdown"],
        "strategy_calmar": ps["calmar"],
        "benchmark_calmar": pb["calmar"],
        "diff_calmar": ps["calmar"] - pb["calmar"],
        "annualized_mean_excess_return": float(excess.mean() * ANNUALIZATION),
        "excess_hit_rate": float((excess > 0).mean()),
    }

def infer_weight_columns(df: pd.DataFrame):
    # Identify ETF/cash weight columns.
    candidates = []
    for c in df.columns:
        lc = str(c).lower()
        if (
            lc.startswith("w_")
            or lc.endswith("_weight")
            or lc in ["0050", "006208", "00692", "00881", "cash"]
            or lc in ["weight_0050", "weight_006208", "weight_00692", "weight_00881", "weight_cash"]
        ):
            candidates.append(c)
    return candidates

def standardize_strategy_column(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "strategy_name" not in out.columns:
        for c in ["policy_name", "strategy", "policy", "name"]:
            if c in out.columns:
                out = out.rename(columns={c: "strategy_name"})
                break
    return out

def find_existing_file(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

# ============================================================
# 4. Load source-aware strict-test matrix
# ============================================================

print("\n" + "=" * 90)
print("Step 1: Loading Notebook 13B source-aware return matrix")
print("=" * 90)

source_aware_mat = standardize_date_index(read_table(NOTEBOOK13B_MATRIX_PATH))
source_aware_mat = source_aware_mat.sort_index()

available_primary = [c for c in PRIMARY_POLICIES_FOR_ROBUSTNESS if c in source_aware_mat.columns]
robustness_base_mat = source_aware_mat[available_primary].copy()

print("Source-aware matrix shape:", source_aware_mat.shape)
print("Date range:", source_aware_mat.index.min(), "to", source_aware_mat.index.max())
print("Available robustness strategies:")
for c in available_primary:
    print(" -", c)

base_perf = build_performance_table(
    robustness_base_mat,
    extra_cols={
        "sensitivity_type": "original_notebook13B",
        "rebalance_frequency": "as_generated",
        "transaction_cost_bps": "as_generated",
    },
)

save_table(
    base_perf,
    local_name="notebook14_00_original_notebook13B_performance.csv",
    global_name=f"table_14_00_original_notebook13B_performance_{RUN_ID}.csv",
)

# ============================================================
# 5. Locate and load weights/returns for true sensitivity
# ============================================================

print("\n" + "=" * 90)
print("Step 2: Loading available strategy weights")
print("=" * 90)

aurora_weight_candidates = [
    AURORA_N10_ROOT / "weights" / "uamv_all_weights.parquet",
    AURORA_N10_ROOT / "weights" / "comparison_weights_uamv_vs_benchmarks.parquet",
    AURORA_N10_ROOT / "weights" / "all_weights.parquet",
    AURORA_N10_ROOT / "weights" / "uamv_weights.parquet",
    AURORA_N10_ROOT / "weights" / "uamv_all_weights.csv",
    AURORA_N10_ROOT / "weights" / "comparison_weights_uamv_vs_benchmarks.csv",
]

roma_weight_candidates = [
    ROMA_R2_ROOT / "weights" / "roma_R2_all_weights.parquet",
    ROMA_R2_ROOT / "weights" / "roma_R2_all_weights.csv",
]

aurora_weight_path = find_existing_file(aurora_weight_candidates)
roma_weight_path = find_existing_file(roma_weight_candidates)

print("AURORA weight path:", aurora_weight_path)
print("ROMA weight path  :", roma_weight_path)

weights_available = False

if aurora_weight_path is not None:
    aurora_weights_raw = read_table(aurora_weight_path)
    aurora_weights_raw = standardize_strategy_column(aurora_weights_raw)
    if "date" in aurora_weights_raw.columns:
        aurora_weights_raw["date"] = pd.to_datetime(aurora_weights_raw["date"])
    else:
        aurora_weights_raw = aurora_weights_raw.reset_index().rename(columns={"index": "date"})
        aurora_weights_raw["date"] = pd.to_datetime(aurora_weights_raw["date"])
else:
    aurora_weights_raw = pd.DataFrame()

if roma_weight_path is not None:
    roma_weights_raw = read_table(roma_weight_path)
    roma_weights_raw = standardize_strategy_column(roma_weights_raw)
    if "date" in roma_weights_raw.columns:
        roma_weights_raw["date"] = pd.to_datetime(roma_weights_raw["date"])
    else:
        roma_weights_raw = roma_weights_raw.reset_index().rename(columns={"index": "date"})
        roma_weights_raw["date"] = pd.to_datetime(roma_weights_raw["date"])
else:
    roma_weights_raw = pd.DataFrame()

print("AURORA weights shape:", aurora_weights_raw.shape)
print("ROMA weights shape  :", roma_weights_raw.shape)

if len(aurora_weights_raw):
    print("AURORA weight columns:", aurora_weights_raw.columns.tolist()[:30])
if len(roma_weights_raw):
    print("ROMA weight columns  :", roma_weights_raw.columns.tolist()[:30])

# Load ETF returns.
if not ETF_RETURN_PANEL_PATH.exists():
    print("ETF return panel missing. True rebalance sensitivity may be limited.")
    etf_returns = pd.DataFrame()
else:
    etf_returns = standardize_date_index(read_table(ETF_RETURN_PANEL_PATH))
    print("ETF return panel shape:", etf_returns.shape)
    print("ETF return panel columns:", etf_returns.columns.tolist())
    print("ETF return date range:", etf_returns.index.min(), "to", etf_returns.index.max())

weights_available = (
    len(etf_returns) > 0
    and (
        len(aurora_weights_raw) > 0
        or len(roma_weights_raw) > 0
    )
)

# ============================================================
# 6. Standardize weights into daily strategy x asset panel
# ============================================================

print("\n" + "=" * 90)
print("Step 3: Standardizing weights")
print("=" * 90)

ETF_ASSETS = ["0050", "006208", "00692", "00881"]
CASH_ASSET = "cash"

def standardize_weights_long(weights_raw: pd.DataFrame, source_prefix: str) -> pd.DataFrame:
    if weights_raw.empty:
        return pd.DataFrame()

    df = weights_raw.copy()
    df = standardize_strategy_column(df)

    if "strategy_name" not in df.columns:
        raise ValueError(f"{source_prefix} weights missing strategy_name/policy_name column.")

    # Standardize asset weight columns.
    rename = {}
    for c in df.columns:
        lc = str(c).lower()
        if lc in ["0050", "w_0050", "weight_0050", "0050_weight"]:
            rename[c] = "0050"
        elif lc in ["006208", "w_006208", "weight_006208", "006208_weight"]:
            rename[c] = "006208"
        elif lc in ["00692", "w_00692", "weight_00692", "00692_weight"]:
            rename[c] = "00692"
        elif lc in ["00881", "w_00881", "weight_00881", "00881_weight"]:
            rename[c] = "00881"
        elif lc in ["cash", "w_cash", "weight_cash", "cash_weight"]:
            rename[c] = "cash"

    df = df.rename(columns=rename)

    asset_cols = [c for c in ETF_ASSETS + [CASH_ASSET] if c in df.columns]

    # If no columns found, try any w_* format.
    if not asset_cols:
        for c in df.columns:
            lc = str(c).lower()
            for asset in ETF_ASSETS + [CASH_ASSET]:
                if asset in lc and ("weight" in lc or lc.startswith("w_")):
                    df = df.rename(columns={c: asset})
        asset_cols = [c for c in ETF_ASSETS + [CASH_ASSET] if c in df.columns]

    if not asset_cols:
        raise ValueError(f"Could not infer asset weight columns for {source_prefix} weights.")

    if "date" not in df.columns:
        raise ValueError(f"{source_prefix} weights missing date column.")

    df["date"] = pd.to_datetime(df["date"])
    df["strategy_name"] = df["strategy_name"].astype(str)

    # Prefix benchmark strategy names only if they are non-AURORA/ROMA strategy names.
    def source_aware_name(x):
        x = str(x)
        if x.startswith("AURORA10_") or x.startswith("ROMA_"):
            return x
        if source_prefix == "AURORA":
            return f"AURORA_{x}"
        if source_prefix == "ROMA":
            return f"ROMA_{x}"
        return x

    df["strategy_name"] = df["strategy_name"].map(source_aware_name)

    keep_cols = ["date", "strategy_name"] + asset_cols
    out = df[keep_cols].copy()

    for asset in ETF_ASSETS + [CASH_ASSET]:
        if asset not in out.columns:
            out[asset] = 0.0

    out = out[["date", "strategy_name"] + ETF_ASSETS + [CASH_ASSET]].copy()
    out[ETF_ASSETS + [CASH_ASSET]] = out[ETF_ASSETS + [CASH_ASSET]].apply(pd.to_numeric, errors="coerce").fillna(0.0)

    # Normalize if sum deviates materially from 1.
    s = out[ETF_ASSETS + [CASH_ASSET]].sum(axis=1)
    nonzero = s.abs() > 1e-12
    out.loc[nonzero, ETF_ASSETS + [CASH_ASSET]] = (
        out.loc[nonzero, ETF_ASSETS + [CASH_ASSET]].div(s.loc[nonzero], axis=0)
    )

    return out

try:
    aurora_weights = standardize_weights_long(aurora_weights_raw, "AURORA")
except Exception as e:
    print("AURORA weight standardization failed:", repr(e))
    aurora_weights = pd.DataFrame()

try:
    roma_weights = standardize_weights_long(roma_weights_raw, "ROMA")
except Exception as e:
    print("ROMA weight standardization failed:", repr(e))
    roma_weights = pd.DataFrame()

all_weights_long = pd.concat([aurora_weights, roma_weights], axis=0, ignore_index=True)

if len(all_weights_long):
    # Filter strict-test dates.
    min_date = source_aware_mat.index.min()
    max_date = source_aware_mat.index.max()
    all_weights_long = all_weights_long[
        (all_weights_long["date"] >= min_date)
        & (all_weights_long["date"] <= max_date)
    ].copy()

print("Standardized weights shape:", all_weights_long.shape)
if len(all_weights_long):
    print("Strategies in weights:")
    print(sorted(all_weights_long["strategy_name"].unique()))

save_table(
    all_weights_long,
    local_name="notebook14_01_standardized_weights_long.csv",
    global_name=f"table_14_01_standardized_weights_long_{RUN_ID}.csv",
)

# ============================================================
# 7. True rebalance/cost sensitivity using weights
# ============================================================

print("\n" + "=" * 90)
print("Step 4: Running true transaction-cost and rebalance sensitivity")
print("=" * 90)

def rebalance_dates_from_index(index: pd.DatetimeIndex, frequency_key: str) -> pd.DatetimeIndex:
    index = pd.DatetimeIndex(index).sort_values()
    if frequency_key == "daily":
        return index

    rule = REBALANCE_FREQUENCIES[frequency_key]
    s = pd.Series(index=index, data=np.arange(len(index)))

    # Take the first trading date in each period.
    if frequency_key == "weekly":
        groups = pd.Series(index=index, data=index).groupby(index.to_period("W-FRI")).first()
    elif frequency_key == "monthly":
        groups = pd.Series(index=index, data=index).groupby(index.to_period("M")).first()
    elif frequency_key == "quarterly":
        groups = pd.Series(index=index, data=index).groupby(index.to_period("Q")).first()
    else:
        raise ValueError(f"Unsupported frequency: {frequency_key}")

    return pd.DatetimeIndex(groups.values).sort_values()

def simulate_strategy_from_weights(
    weights_long: pd.DataFrame,
    strategy_name: str,
    etf_returns: pd.DataFrame,
    dates: pd.DatetimeIndex,
    transaction_cost_bps: float,
    rebalance_frequency: str,
) -> pd.DataFrame:
    # Prepare return panel.
    ret = etf_returns.copy()
    ret.index = pd.to_datetime(ret.index)
    ret = ret.sort_index()
    available_assets = [a for a in ETF_ASSETS if a in ret.columns]
    if not available_assets:
        raise ValueError("No ETF asset columns found in return panel.")

    # Align dates.
    common = pd.DatetimeIndex(dates).intersection(ret.index).sort_values()

    # Prepare weights for this strategy.
    w = weights_long[weights_long["strategy_name"] == strategy_name].copy()
    if w.empty:
        raise ValueError(f"No weights for strategy: {strategy_name}")

    w["date"] = pd.to_datetime(w["date"])
    w = w.sort_values("date")
    w = w.set_index("date")

    # Keep only strict-test dates, forward-fill from available weights.
    w = w.reindex(common).ffill().bfill()

    # Ensure all columns exist.
    for asset in ETF_ASSETS + [CASH_ASSET]:
        if asset not in w.columns:
            w[asset] = 0.0

    # Normalize.
    s = w[ETF_ASSETS + [CASH_ASSET]].sum(axis=1)
    nonzero = s.abs() > 1e-12
    w.loc[nonzero, ETF_ASSETS + [CASH_ASSET]] = w.loc[nonzero, ETF_ASSETS + [CASH_ASSET]].div(s.loc[nonzero], axis=0)

    rb_dates = rebalance_dates_from_index(common, rebalance_frequency)
    rb_mask = pd.Series(False, index=common)
    rb_mask.loc[rb_dates] = True

    # Holding weights: update only at rebalance dates.
    target_w = w[ETF_ASSETS + [CASH_ASSET]].copy()

    hold_w = pd.DataFrame(index=common, columns=ETF_ASSETS + [CASH_ASSET], dtype=float)
    last_w = None

    for dt in common:
        if last_w is None:
            last_w = target_w.loc[dt].copy()
        elif rb_mask.loc[dt]:
            last_w = target_w.loc[dt].copy()
        hold_w.loc[dt] = last_w.values

    hold_w = hold_w.astype(float).fillna(0.0)

    # Portfolio gross return uses beginning-of-day/held weight.
    asset_ret = ret.reindex(common)[ETF_ASSETS].fillna(0.0)
    gross_return = (hold_w[ETF_ASSETS].values * asset_ret[ETF_ASSETS].values).sum(axis=1)
    gross_return = pd.Series(gross_return, index=common)

    # Turnover applied only when rebalanced.
    target_for_turnover = hold_w.copy()
    prev_w = hold_w.shift(1).fillna(0.0)
    turnover = (hold_w[ETF_ASSETS + [CASH_ASSET]] - prev_w[ETF_ASSETS + [CASH_ASSET]]).abs().sum(axis=1)
    turnover.loc[~rb_mask] = 0.0

    tc_rate = transaction_cost_bps / 10000.0
    transaction_cost = turnover * tc_rate
    net_return = gross_return - transaction_cost

    out = pd.DataFrame({
        "date": common,
        "strategy_name": strategy_name,
        "rebalance_frequency": rebalance_frequency,
        "transaction_cost_bps": transaction_cost_bps,
        "gross_return": gross_return.values,
        "turnover": turnover.values,
        "transaction_cost": transaction_cost.values,
        "net_return": net_return.values,
        "is_rebalance_date": rb_mask.values,
    })

    for asset in ETF_ASSETS + [CASH_ASSET]:
        out[f"weight_{asset}"] = hold_w[asset].values

    out["equity"] = (1.0 + out["net_return"]).cumprod()
    out["drawdown"] = out["equity"] / out["equity"].cummax() - 1.0

    return out

true_sensitivity_rows = []
true_sensitivity_return_mats = {}

strategies_with_weights = sorted(all_weights_long["strategy_name"].unique()) if len(all_weights_long) else []
strategies_to_simulate = [s for s in PRIMARY_POLICIES_FOR_ROBUSTNESS if s in strategies_with_weights]

print("Strategies with true weights to simulate:")
for s in strategies_to_simulate:
    print(" -", s)

if len(strategies_to_simulate) == 0:
    print("No strategy weights available for true sensitivity simulation.")
else:
    strict_dates = source_aware_mat.index

    for freq in REBALANCE_FREQUENCIES.keys():
        for tc_bps in TRANSACTION_COST_BPS_GRID:
            returns_this_setting = pd.DataFrame(index=strict_dates)

            for strategy in strategies_to_simulate:
                try:
                    sim = simulate_strategy_from_weights(
                        weights_long=all_weights_long,
                        strategy_name=strategy,
                        etf_returns=etf_returns,
                        dates=strict_dates,
                        transaction_cost_bps=tc_bps,
                        rebalance_frequency=freq,
                    )

                    sim_path_base = (
                        RETURN_RUN_DIR
                        / f"notebook14_true_sim_{strategy}_{freq}_{tc_bps}bps"
                    )
                    save_parquet_and_csv(sim, sim_path_base, index=False)

                    r = sim.set_index("date")["net_return"]
                    returns_this_setting[strategy] = r.reindex(strict_dates)

                except Exception as e:
                    print(f"Simulation failed: {strategy}, {freq}, {tc_bps} bps:", repr(e))

            if len(returns_this_setting.columns):
                key = f"{freq}_{tc_bps}bps"
                true_sensitivity_return_mats[key] = returns_this_setting.copy()

                perf = build_performance_table(
                    returns_this_setting,
                    extra_cols={
                        "sensitivity_type": "true_weight_resimulation",
                        "rebalance_frequency": freq,
                        "transaction_cost_bps": tc_bps,
                    },
                )
                true_sensitivity_rows.append(perf)

                base_path = RETURN_RUN_DIR / f"notebook14_true_weight_resimulation_returns_{key}"
                save_parquet_and_csv(returns_this_setting, base_path, index=True)

if true_sensitivity_rows:
    true_sensitivity_perf = pd.concat(true_sensitivity_rows, axis=0, ignore_index=True)
else:
    true_sensitivity_perf = pd.DataFrame()

save_table(
    true_sensitivity_perf,
    local_name="notebook14_02_true_weight_resimulation_performance.csv",
    global_name=f"table_14_02_true_weight_resimulation_performance_{RUN_ID}.csv",
)

print("True sensitivity performance shape:", true_sensitivity_perf.shape)

# ============================================================
# 8. Fallback approximate sensitivity from return series
# ============================================================

print("\n" + "=" * 90)
print("Step 5: Running approximate sensitivity from return series")
print("=" * 90)

def approximate_rebalance_frequency_returns(
    r: pd.Series,
    frequency_key: str,
) -> pd.Series:
    """
    Approximate lower-frequency rebalancing when weights are unavailable.

    This does NOT reconstruct actual portfolio weights.
    It is a conservative diagnostic transformation:
    - daily: original returns
    - weekly/monthly/quarterly: compound original returns within periods,
      then spread the period compound return equally across trading days.

    This keeps total period return similar but smooths high-frequency turnover effects.
    Use only as fallback diagnostic if true weights are unavailable.
    """
    r = pd.Series(r).dropna().astype(float)
    r.index = pd.to_datetime(r.index)

    if frequency_key == "daily":
        return r.copy()

    if frequency_key == "weekly":
        period = r.index.to_period("W-FRI")
    elif frequency_key == "monthly":
        period = r.index.to_period("M")
    elif frequency_key == "quarterly":
        period = r.index.to_period("Q")
    else:
        raise ValueError(f"Unsupported frequency: {frequency_key}")

    out = pd.Series(index=r.index, dtype=float)

    for p, idx in pd.Series(r.index, index=r.index).groupby(period):
        dates = pd.DatetimeIndex(idx.values)
        rr = r.loc[dates]
        compounded = float((1.0 + rr).prod() - 1.0)
        daily_equiv = (1.0 + compounded) ** (1.0 / len(rr)) - 1.0
        out.loc[dates] = daily_equiv

    return out

def estimate_turnover_proxy(r: pd.Series, frequency_key: str) -> pd.Series:
    """
    Approximate turnover proxy when actual weights are unavailable.
    This is only for sensitivity diagnostics.
    """
    r = pd.Series(r).dropna().astype(float)
    idx = r.index

    if frequency_key == "daily":
        rb_dates = idx
    else:
        rb_dates = rebalance_dates_from_index(idx, frequency_key)

    turnover = pd.Series(0.0, index=idx)
    # Conservative proxy: 0.5 turnover on rebalance date for active strategies.
    turnover.loc[rb_dates] = 0.5
    return turnover

approx_rows = []
approx_return_mats = {}

for freq in REBALANCE_FREQUENCIES.keys():
    for tc_bps in TRANSACTION_COST_BPS_GRID:
        mat = pd.DataFrame(index=robustness_base_mat.index)

        for strategy in robustness_base_mat.columns:
            r0 = robustness_base_mat[strategy].dropna()
            r_freq = approximate_rebalance_frequency_returns(r0, freq)
            turnover_proxy = estimate_turnover_proxy(r_freq, freq)
            tc_rate = tc_bps / 10000.0
            net = r_freq - turnover_proxy * tc_rate
            mat[strategy] = net.reindex(robustness_base_mat.index)

        key = f"{freq}_{tc_bps}bps"
        approx_return_mats[key] = mat.copy()

        perf = build_performance_table(
            mat,
            extra_cols={
                "sensitivity_type": "approx_return_series_transformation",
                "rebalance_frequency": freq,
                "transaction_cost_bps": tc_bps,
            },
        )
        approx_rows.append(perf)

        base_path = RETURN_RUN_DIR / f"notebook14_approx_return_series_returns_{key}"
        save_parquet_and_csv(mat, base_path, index=True)

approx_perf = pd.concat(approx_rows, axis=0, ignore_index=True)

save_table(
    approx_perf,
    local_name="notebook14_03_approx_return_series_performance.csv",
    global_name=f"table_14_03_approx_return_series_performance_{RUN_ID}.csv",
)

print("Approximate sensitivity performance shape:", approx_perf.shape)

# ============================================================
# 9. Choose primary sensitivity table
# ============================================================

print("\n" + "=" * 90)
print("Step 6: Selecting primary sensitivity evidence")
print("=" * 90)

if len(true_sensitivity_perf):
    primary_sensitivity_perf = true_sensitivity_perf.copy()
    primary_sensitivity_source = "true_weight_resimulation"
    primary_return_mats = true_sensitivity_return_mats
else:
    primary_sensitivity_perf = approx_perf.copy()
    primary_sensitivity_source = "approx_return_series_transformation"
    primary_return_mats = approx_return_mats

print("Primary sensitivity source:", primary_sensitivity_source)
print("Primary sensitivity shape :", primary_sensitivity_perf.shape)

save_table(
    primary_sensitivity_perf,
    local_name="notebook14_04_primary_sensitivity_performance.csv",
    global_name=f"table_14_04_primary_sensitivity_performance_{RUN_ID}.csv",
)

# ============================================================
# 10. Robustness summary: AURORA vs ROMA and benchmarks
# ============================================================

print("\n" + "=" * 90)
print("Step 7: Building robustness pairwise summary")
print("=" * 90)

COMPARISON_PAIRS = [
    (PRIMARY_AURORA_POLICY, PRIMARY_ROMA_POLICY, "AURORA10-UAMV-B vs ROMA-P4"),
    (PRIMARY_AURORA_POLICY, "ROMA_P2_20d_only_return_seeking", "AURORA10-UAMV-B vs ROMA-P2"),
    (PRIMARY_AURORA_POLICY, "ROMA_B12_ma_timing_equal_weight", "AURORA10-UAMV-B vs ROMA-B12"),
    (PRIMARY_AURORA_POLICY, "ROMA_B6_00881_only", "AURORA10-UAMV-B vs ROMA-B6"),
    (PRIMARY_AURORA_POLICY, "ROMA_B3_0050_only", "AURORA10-UAMV-B vs ROMA-B3"),
    (PRIMARY_AURORA_POLICY, "ROMA_B1_equal_weight_all_etfs", "AURORA10-UAMV-B vs ROMA-B1"),
]

pairwise_rows = []

# Use primary_return_mats when available. If true matrices only include strategies with weights,
# some benchmark comparisons may be unavailable. Add original 13B comparisons separately.
for key, mat in primary_return_mats.items():
    for strategy, benchmark, label in COMPARISON_PAIRS:
        if strategy in mat.columns and benchmark in mat.columns:
            diff = paired_difference(mat[strategy], mat[benchmark])
            freq, tc = key.rsplit("_", 1)
            tc_bps = float(tc.replace("bps", ""))
            pairwise_rows.append({
                "sensitivity_source": primary_sensitivity_source,
                "scenario_key": key,
                "rebalance_frequency": freq,
                "transaction_cost_bps": tc_bps,
                "comparison_label": label,
                "strategy_name": strategy,
                "strategy_display_name": DISPLAY_NAMES.get(strategy, strategy),
                "benchmark_name": benchmark,
                "benchmark_display_name": DISPLAY_NAMES.get(benchmark, benchmark),
                **diff,
            })

primary_pairwise = pd.DataFrame(pairwise_rows)

save_table(
    primary_pairwise,
    local_name="notebook14_05_primary_sensitivity_pairwise_differences.csv",
    global_name=f"table_14_05_primary_sensitivity_pairwise_differences_{RUN_ID}.csv",
)

print("Primary pairwise sensitivity shape:", primary_pairwise.shape)
if len(primary_pairwise):
    print(
        primary_pairwise[
            [
                "rebalance_frequency",
                "transaction_cost_bps",
                "comparison_label",
                "diff_total_return",
                "diff_sharpe",
                "diff_sortino",
                "drawdown_improvement",
                "diff_calmar",
            ]
        ].head(40).to_string(index=False)
    )

# ============================================================
# 11. Scenario-level claim stability table
# ============================================================

print("\n" + "=" * 90)
print("Step 8: Building claim stability table")
print("=" * 90)

claim_rows = []

def get_perf_row(perf_df, strategy, freq, tc_bps):
    rows = perf_df[
        (perf_df["strategy_name"] == strategy)
        & (perf_df["rebalance_frequency"] == freq)
        & (perf_df["transaction_cost_bps"].astype(float) == float(tc_bps))
    ]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()

for freq in REBALANCE_FREQUENCIES.keys():
    for tc_bps in TRANSACTION_COST_BPS_GRID:
        aur = get_perf_row(primary_sensitivity_perf, PRIMARY_AURORA_POLICY, freq, tc_bps)
        roma = get_perf_row(primary_sensitivity_perf, PRIMARY_ROMA_POLICY, freq, tc_bps)
        roma_b12 = get_perf_row(primary_sensitivity_perf, "ROMA_B12_ma_timing_equal_weight", freq, tc_bps)

        row = {
            "sensitivity_source": primary_sensitivity_source,
            "rebalance_frequency": freq,
            "transaction_cost_bps": tc_bps,
            "has_aurora": aur is not None,
            "has_roma_p4": roma is not None,
            "has_roma_b12": roma_b12 is not None,
        }

        if aur is not None:
            row.update({
                "aurora_total_return": aur["total_return"],
                "aurora_sharpe": aur["sharpe"],
                "aurora_sortino": aur["sortino"],
                "aurora_max_drawdown": aur["max_drawdown"],
                "aurora_calmar": aur["calmar"],
            })

        if roma is not None:
            row.update({
                "roma_p4_total_return": roma["total_return"],
                "roma_p4_sharpe": roma["sharpe"],
                "roma_p4_sortino": roma["sortino"],
                "roma_p4_max_drawdown": roma["max_drawdown"],
                "roma_p4_calmar": roma["calmar"],
            })

        if roma_b12 is not None:
            row.update({
                "roma_b12_total_return": roma_b12["total_return"],
                "roma_b12_sharpe": roma_b12["sharpe"],
                "roma_b12_sortino": roma_b12["sortino"],
                "roma_b12_max_drawdown": roma_b12["max_drawdown"],
                "roma_b12_calmar": roma_b12["calmar"],
            })

        if aur is not None and roma is not None:
            row.update({
                "aurora_minus_roma_total_return": aur["total_return"] - roma["total_return"],
                "aurora_minus_roma_sharpe": aur["sharpe"] - roma["sharpe"],
                "aurora_minus_roma_sortino": aur["sortino"] - roma["sortino"],
                "aurora_drawdown_improvement_vs_roma": aur["max_drawdown"] - roma["max_drawdown"],
                "aurora_sharpe_gt_roma": aur["sharpe"] > roma["sharpe"],
                "aurora_sortino_gt_roma": aur["sortino"] > roma["sortino"],
                "aurora_mdd_less_severe_than_roma": aur["max_drawdown"] > roma["max_drawdown"],
                "aurora_total_return_gt_roma": aur["total_return"] > roma["total_return"],
            })

        if roma is not None and roma_b12 is not None:
            row.update({
                "roma_minus_b12_total_return": roma["total_return"] - roma_b12["total_return"],
                "roma_minus_b12_sharpe": roma["sharpe"] - roma_b12["sharpe"],
                "roma_drawdown_improvement_vs_b12": roma["max_drawdown"] - roma_b12["max_drawdown"],
                "roma_sharpe_gt_b12": roma["sharpe"] > roma_b12["sharpe"],
                "roma_mdd_less_severe_than_b12": roma["max_drawdown"] > roma_b12["max_drawdown"],
                "roma_total_return_gt_b12": roma["total_return"] > roma_b12["total_return"],
            })

        claim_rows.append(row)

claim_stability = pd.DataFrame(claim_rows)

# Summarize Boolean stability.
bool_cols = [
    "aurora_sharpe_gt_roma",
    "aurora_sortino_gt_roma",
    "aurora_mdd_less_severe_than_roma",
    "aurora_total_return_gt_roma",
    "roma_sharpe_gt_b12",
    "roma_mdd_less_severe_than_b12",
    "roma_total_return_gt_b12",
]

summary_rows = []
for c in bool_cols:
    if c in claim_stability.columns:
        valid = claim_stability[c].dropna()
        summary_rows.append({
            "claim_indicator": c,
            "n_scenarios_available": int(len(valid)),
            "n_true": int(valid.sum()) if len(valid) else 0,
            "share_true": float(valid.mean()) if len(valid) else np.nan,
        })

claim_stability_summary = pd.DataFrame(summary_rows)

save_table(
    claim_stability,
    local_name="notebook14_06_claim_stability_by_scenario.csv",
    global_name=f"table_14_06_claim_stability_by_scenario_{RUN_ID}.csv",
)

save_table(
    claim_stability_summary,
    local_name="notebook14_07_claim_stability_summary.csv",
    global_name=f"table_14_07_claim_stability_summary_{RUN_ID}.csv",
)

print("Claim stability summary:")
print(claim_stability_summary.to_string(index=False))

# ============================================================
# 12. Paper-ready robustness tables
# ============================================================

print("\n" + "=" * 90)
print("Step 9: Creating paper-ready robustness tables")
print("=" * 90)

# Table A: AURORA and ROMA metrics across scenarios.
paper_metrics = [
    PRIMARY_AURORA_POLICY,
    PRIMARY_ROMA_POLICY,
    "ROMA_P2_20d_only_return_seeking",
    "ROMA_B12_ma_timing_equal_weight",
    "ROMA_B6_00881_only",
    "ROMA_B3_0050_only",
    "ROMA_B1_equal_weight_all_etfs",
]

paper_sensitivity = primary_sensitivity_perf[
    primary_sensitivity_perf["strategy_name"].isin(paper_metrics)
].copy()

for c in [
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe",
    "sortino",
    "max_drawdown",
    "calmar",
    "composite_rank",
]:
    if c in paper_sensitivity.columns:
        paper_sensitivity[c] = paper_sensitivity[c].astype(float).round(4)

save_table(
    paper_sensitivity,
    local_name="notebook14_08_paper_robustness_performance_table.csv",
    global_name=f"table_14_08_paper_robustness_performance_table_{RUN_ID}.csv",
)

# Table B: AURORA vs ROMA pairwise across scenarios.
paper_pairwise = primary_pairwise[
    primary_pairwise["comparison_label"].isin([
        "AURORA10-UAMV-B vs ROMA-P4",
        "AURORA10-UAMV-B vs ROMA-P2",
        "AURORA10-UAMV-B vs ROMA-B12",
    ])
].copy()

for c in [
    "diff_total_return",
    "diff_annual_return",
    "diff_sharpe",
    "diff_sortino",
    "drawdown_improvement",
    "diff_calmar",
    "annualized_mean_excess_return",
]:
    if c in paper_pairwise.columns:
        paper_pairwise[c] = paper_pairwise[c].astype(float).round(4)

save_table(
    paper_pairwise,
    local_name="notebook14_09_paper_robustness_pairwise_table.csv",
    global_name=f"table_14_09_paper_robustness_pairwise_table_{RUN_ID}.csv",
)

print("Paper robustness performance table preview:")
print(
    paper_sensitivity[
        [
            "rebalance_frequency",
            "transaction_cost_bps",
            "display_name",
            "total_return",
            "sharpe",
            "sortino",
            "max_drawdown",
            "calmar",
        ]
    ].head(60).to_string(index=False)
)

print("\nPaper robustness pairwise table preview:")
if len(paper_pairwise):
    print(
        paper_pairwise[
            [
                "rebalance_frequency",
                "transaction_cost_bps",
                "comparison_label",
                "diff_total_return",
                "diff_sharpe",
                "diff_sortino",
                "drawdown_improvement",
                "diff_calmar",
            ]
        ].head(60).to_string(index=False)
    )
else:
    print("Pairwise table empty for selected comparisons.")

# ============================================================
# 13. Figures
# ============================================================

print("\n" + "=" * 90)
print("Step 10: Creating robustness figures")
print("=" * 90)

figure_records = []

def save_figure_record(path, figure_id, title, caption):
    figure_records.append({
        "figure_id": figure_id,
        "path": str(path),
        "title": title,
        "caption": caption,
    })

# Figure 14.1: AURORA metrics across cost/rebalance.
aurora_scen = primary_sensitivity_perf[
    primary_sensitivity_perf["strategy_name"] == PRIMARY_AURORA_POLICY
].copy()

if len(aurora_scen):
    pivot = aurora_scen.pivot_table(
        index="rebalance_frequency",
        columns="transaction_cost_bps",
        values="sharpe",
        aggfunc="first",
    )
    pivot = pivot.reindex(list(REBALANCE_FREQUENCIES.keys()))

    plt.figure(figsize=(8, 5))
    if HAS_SEABORN:
        sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Blues")
    else:
        plt.imshow(pivot.values, aspect="auto")
        plt.colorbar()
        plt.xticks(range(len(pivot.columns)), pivot.columns)
        plt.yticks(range(len(pivot.index)), pivot.index)

    plt.title("AURORA10-UAMV-B Sharpe robustness")
    plt.xlabel("Transaction cost, bps")
    plt.ylabel("Rebalance frequency")
    plt.tight_layout()

    fig_path = PAPER_FIGURE_DIR / "figure14_01_aurora_sharpe_robustness_heatmap.png"
    plt.savefig(fig_path, dpi=220)
    plt.close()

    save_figure_record(
        fig_path,
        "Figure 14.1",
        "AURORA10-UAMV-B Sharpe robustness",
        "Sharpe ratio of AURORA10-UAMV-B across transaction-cost and rebalance-frequency scenarios.",
    )

# Figure 14.2: AURORA max drawdown robustness.
if len(aurora_scen):
    pivot = aurora_scen.pivot_table(
        index="rebalance_frequency",
        columns="transaction_cost_bps",
        values="max_drawdown",
        aggfunc="first",
    )
    pivot = pivot.reindex(list(REBALANCE_FREQUENCIES.keys()))

    plt.figure(figsize=(8, 5))
    if HAS_SEABORN:
        sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn", center=0)
    else:
        plt.imshow(pivot.values, aspect="auto")
        plt.colorbar()
        plt.xticks(range(len(pivot.columns)), pivot.columns)
        plt.yticks(range(len(pivot.index)), pivot.index)

    plt.title("AURORA10-UAMV-B max drawdown robustness")
    plt.xlabel("Transaction cost, bps")
    plt.ylabel("Rebalance frequency")
    plt.tight_layout()

    fig_path = PAPER_FIGURE_DIR / "figure14_02_aurora_max_drawdown_robustness_heatmap.png"
    plt.savefig(fig_path, dpi=220)
    plt.close()

    save_figure_record(
        fig_path,
        "Figure 14.2",
        "AURORA10-UAMV-B maximum-drawdown robustness",
        "Maximum drawdown of AURORA10-UAMV-B across transaction-cost and rebalance-frequency scenarios. Less negative values indicate smaller drawdowns.",
    )

# Figure 14.3: AURORA vs ROMA Sharpe difference.
if len(primary_pairwise):
    pair = primary_pairwise[
        primary_pairwise["comparison_label"] == "AURORA10-UAMV-B vs ROMA-P4"
    ].copy()

    if len(pair):
        pivot = pair.pivot_table(
            index="rebalance_frequency",
            columns="transaction_cost_bps",
            values="diff_sharpe",
            aggfunc="first",
        )
        pivot = pivot.reindex(list(REBALANCE_FREQUENCIES.keys()))

        plt.figure(figsize=(8, 5))
        if HAS_SEABORN:
            sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdBu", center=0)
        else:
            plt.imshow(pivot.values, aspect="auto")
            plt.colorbar()
            plt.xticks(range(len(pivot.columns)), pivot.columns)
            plt.yticks(range(len(pivot.index)), pivot.index)

        plt.title("AURORA minus ROMA-P4 Sharpe robustness")
        plt.xlabel("Transaction cost, bps")
        plt.ylabel("Rebalance frequency")
        plt.tight_layout()

        fig_path = PAPER_FIGURE_DIR / "figure14_03_aurora_minus_roma_sharpe_heatmap.png"
        plt.savefig(fig_path, dpi=220)
        plt.close()

        save_figure_record(
            fig_path,
            "Figure 14.3",
            "AURORA versus ROMA-P4 Sharpe robustness",
            "Difference in Sharpe ratio between AURORA10-UAMV-B and ROMA-P4 across sensitivity scenarios. Positive values favor AURORA.",
        )

# Figure 14.4: AURORA vs ROMA drawdown improvement.
if len(primary_pairwise):
    pair = primary_pairwise[
        primary_pairwise["comparison_label"] == "AURORA10-UAMV-B vs ROMA-P4"
    ].copy()

    if len(pair):
        pivot = pair.pivot_table(
            index="rebalance_frequency",
            columns="transaction_cost_bps",
            values="drawdown_improvement",
            aggfunc="first",
        )
        pivot = pivot.reindex(list(REBALANCE_FREQUENCIES.keys()))

        plt.figure(figsize=(8, 5))
        if HAS_SEABORN:
            sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdBu", center=0)
        else:
            plt.imshow(pivot.values, aspect="auto")
            plt.colorbar()
            plt.xticks(range(len(pivot.columns)), pivot.columns)
            plt.yticks(range(len(pivot.index)), pivot.index)

        plt.title("AURORA minus ROMA-P4 drawdown improvement")
        plt.xlabel("Transaction cost, bps")
        plt.ylabel("Rebalance frequency")
        plt.tight_layout()

        fig_path = PAPER_FIGURE_DIR / "figure14_04_aurora_minus_roma_drawdown_heatmap.png"
        plt.savefig(fig_path, dpi=220)
        plt.close()

        save_figure_record(
            fig_path,
            "Figure 14.4",
            "AURORA versus ROMA-P4 drawdown robustness",
            "Maximum-drawdown improvement of AURORA10-UAMV-B relative to ROMA-P4 across sensitivity scenarios. Positive values favor AURORA.",
        )

# Figure 14.5: Equity curves for selected scenario.
selected_scenario_key = "monthly_25bps"
if selected_scenario_key not in primary_return_mats:
    selected_scenario_key = next(iter(primary_return_mats.keys())) if primary_return_mats else None

if selected_scenario_key is not None:
    mat = primary_return_mats[selected_scenario_key]
    plot_cols = [
        PRIMARY_AURORA_POLICY,
        PRIMARY_ROMA_POLICY,
        "ROMA_B12_ma_timing_equal_weight",
        "ROMA_B6_00881_only",
        "ROMA_B1_equal_weight_all_etfs",
    ]
    plot_cols = [c for c in plot_cols if c in mat.columns]

    if plot_cols:
        plt.figure(figsize=(11, 6))
        for c in plot_cols:
            eq = equity_from_returns(mat[c])
            plt.plot(eq.index, eq.values, linewidth=1.8, label=DISPLAY_NAMES.get(c, c))

        plt.title(f"Robustness scenario equity curves: {selected_scenario_key}")
        plt.xlabel("Date")
        plt.ylabel("Equity")
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=9)
        plt.tight_layout()

        fig_path = PAPER_FIGURE_DIR / f"figure14_05_equity_curves_{selected_scenario_key}.png"
        plt.savefig(fig_path, dpi=220)
        plt.close()

        save_figure_record(
            fig_path,
            "Figure 14.5",
            f"Equity curves under robustness scenario {selected_scenario_key}",
            "Equity curves for selected methods under a representative transaction-cost and rebalance-frequency robustness scenario.",
        )

# Figure 14.6: Claim stability bar.
if len(claim_stability_summary):
    plot_df = claim_stability_summary.copy()
    plt.figure(figsize=(10, 5))
    plt.barh(plot_df["claim_indicator"], plot_df["share_true"], color="#4C78A8")
    plt.xlabel("Share of scenarios where indicator is true")
    plt.xlim(0, 1)
    plt.title("Claim stability across robustness scenarios")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    fig_path = PAPER_FIGURE_DIR / "figure14_06_claim_stability_bar.png"
    plt.savefig(fig_path, dpi=220)
    plt.close()

    save_figure_record(
        fig_path,
        "Figure 14.6",
        "Claim stability across robustness scenarios",
        "Share of transaction-cost and rebalance-frequency scenarios in which each claim indicator holds.",
    )

figure_index = pd.DataFrame(figure_records)

save_table(
    figure_index,
    local_name="notebook14_10_figure_index.csv",
    global_name=f"table_14_10_figure_index_{RUN_ID}.csv",
)

print("Figure index:")
print(figure_index.to_string(index=False))

# ============================================================
# 14. Manuscript text assets
# ============================================================

print("\n" + "=" * 90)
print("Step 11: Writing manuscript text assets")
print("=" * 90)

def summarize_claim_share(indicator):
    rows = claim_stability_summary[claim_stability_summary["claim_indicator"] == indicator]
    if rows.empty:
        return "NA"
    r = rows.iloc[0]
    return f"{int(r['n_true'])}/{int(r['n_scenarios_available'])} ({100*r['share_true']:.1f}%)"

aurora_sharpe_stability = summarize_claim_share("aurora_sharpe_gt_roma")
aurora_sortino_stability = summarize_claim_share("aurora_sortino_gt_roma")
aurora_dd_stability = summarize_claim_share("aurora_mdd_less_severe_than_roma")
aurora_return_stability = summarize_claim_share("aurora_total_return_gt_roma")

robustness_methods_text = f"""
## Notebook 14 robustness-methods text

To evaluate whether the main findings are sensitive to implementation assumptions, we performed a transaction-cost and rebalance-frequency robustness analysis. The analysis used the source-aware strict-test return matrix from Notebook 13B and, where available, the underlying strategy weights from the AURORA and ROMA allocation notebooks. We evaluated transaction costs of 0, 10, 25, and 50 basis points and rebalance frequencies of daily, weekly, monthly, and quarterly. The primary method remained AURORA10-UAMV-B; no new method was selected from the robustness grid.

When strategy weights were available, returns were re-simulated by holding weights between rebalance dates and subtracting transaction costs proportional to turnover. When weights were not available, an approximate return-series transformation was used only as a diagnostic fallback. The primary sensitivity source for this run was: `{primary_sensitivity_source}`.
""".strip()

robustness_results_text = f"""
## Notebook 14 robustness-results text

Across the robustness grid, the AURORA-versus-ROMA risk-control claim was evaluated by checking whether AURORA10-UAMV-B exceeded ROMA-P4 in Sharpe, Sortino, and maximum-drawdown behavior. The indicator that AURORA had higher Sharpe than ROMA-P4 held in {aurora_sharpe_stability} available scenarios. The indicator that AURORA had higher Sortino held in {aurora_sortino_stability} available scenarios. The indicator that AURORA had a less severe maximum drawdown held in {aurora_dd_stability} available scenarios. By contrast, the indicator that AURORA had higher total return than ROMA-P4 held in {aurora_return_stability} available scenarios.

These results should be interpreted as a robustness check, not as a new model-selection stage. The main paper claim remains that AURORA10-UAMV-B is a downside-risk and risk-adjusted allocation layer, not a total-return-dominant trading rule.
""".strip()

robustness_limitations_text = """
## Notebook 14 robustness-limitations text

This robustness analysis is limited by the availability and format of stored strategy weights. When complete daily weights are available, transaction-cost and rebalance-frequency scenarios can be directly re-simulated. When weights are unavailable, return-series transformations provide only diagnostic approximations and should not be treated as exact rebalanced portfolio simulations. Therefore, the paper should clearly state which robustness tables are based on true weight re-simulation and which, if any, are approximate diagnostics.
""".strip()

robustness_claim_text = """
## Recommended paper wording

The transaction-cost and rebalance-frequency sensitivity analysis supports the interpretation of AURORA as a downside-risk-control method. Across alternative cost and rebalance assumptions, the robustness analysis evaluates whether the main risk-adjusted comparison against ROMA-P4 is preserved. This analysis is not used to choose a new best-performing strategy; it is used only to test whether the main conclusion is fragile to implementation assumptions.
""".strip()

write_markdown(MANUSCRIPT_RUN_DIR / "notebook14_robustness_methods.md", robustness_methods_text)
write_markdown(MANUSCRIPT_RUN_DIR / "notebook14_robustness_results.md", robustness_results_text)
write_markdown(MANUSCRIPT_RUN_DIR / "notebook14_robustness_limitations.md", robustness_limitations_text)
write_markdown(MANUSCRIPT_RUN_DIR / "notebook14_recommended_paper_wording.md", robustness_claim_text)

write_markdown(GLOBAL_MANUSCRIPT_DIR / f"{RUN_ID}_notebook14_robustness_methods.md", robustness_methods_text)
write_markdown(GLOBAL_MANUSCRIPT_DIR / f"{RUN_ID}_notebook14_robustness_results.md", robustness_results_text)
write_markdown(GLOBAL_MANUSCRIPT_DIR / f"{RUN_ID}_notebook14_robustness_limitations.md", robustness_limitations_text)
write_markdown(GLOBAL_MANUSCRIPT_DIR / f"{RUN_ID}_notebook14_recommended_paper_wording.md", robustness_claim_text)

# ============================================================
# 15. Output index
# ============================================================

print("\n" + "=" * 90)
print("Step 12: Creating output index")
print("=" * 90)

output_rows = [
    {
        "artifact_type": "table",
        "name": "original_notebook13B_performance",
        "path": str(TABLE_RUN_DIR / "notebook14_00_original_notebook13B_performance.csv"),
        "description": "Original Notebook 13B performance for selected robustness strategies.",
    },
    {
        "artifact_type": "table",
        "name": "standardized_weights_long",
        "path": str(TABLE_RUN_DIR / "notebook14_01_standardized_weights_long.csv"),
        "description": "Standardized strategy weights used for true re-simulation when available.",
    },
    {
        "artifact_type": "table",
        "name": "true_weight_resimulation_performance",
        "path": str(TABLE_RUN_DIR / "notebook14_02_true_weight_resimulation_performance.csv"),
        "description": "Performance under transaction-cost and rebalance-frequency scenarios using true weights where available.",
    },
    {
        "artifact_type": "table",
        "name": "approx_return_series_performance",
        "path": str(TABLE_RUN_DIR / "notebook14_03_approx_return_series_performance.csv"),
        "description": "Fallback approximate sensitivity performance from return-series transformations.",
    },
    {
        "artifact_type": "table",
        "name": "primary_sensitivity_performance",
        "path": str(TABLE_RUN_DIR / "notebook14_04_primary_sensitivity_performance.csv"),
        "description": "Primary robustness performance table selected for interpretation.",
    },
    {
        "artifact_type": "table",
        "name": "primary_sensitivity_pairwise_differences",
        "path": str(TABLE_RUN_DIR / "notebook14_05_primary_sensitivity_pairwise_differences.csv"),
        "description": "Pairwise performance differences across robustness scenarios.",
    },
    {
        "artifact_type": "table",
        "name": "claim_stability_by_scenario",
        "path": str(TABLE_RUN_DIR / "notebook14_06_claim_stability_by_scenario.csv"),
        "description": "Scenario-level indicators for claim stability.",
    },
    {
        "artifact_type": "table",
        "name": "claim_stability_summary",
        "path": str(TABLE_RUN_DIR / "notebook14_07_claim_stability_summary.csv"),
        "description": "Summary of how often each claim indicator holds across scenarios.",
    },
    {
        "artifact_type": "table",
        "name": "paper_robustness_performance_table",
        "path": str(TABLE_RUN_DIR / "notebook14_08_paper_robustness_performance_table.csv"),
        "description": "Paper-ready robustness performance table.",
    },
    {
        "artifact_type": "table",
        "name": "paper_robustness_pairwise_table",
        "path": str(TABLE_RUN_DIR / "notebook14_09_paper_robustness_pairwise_table.csv"),
        "description": "Paper-ready robustness pairwise table.",
    },
    {
        "artifact_type": "table",
        "name": "figure_index",
        "path": str(TABLE_RUN_DIR / "notebook14_10_figure_index.csv"),
        "description": "Index of robustness figures.",
    },
    {
        "artifact_type": "manuscript",
        "name": "robustness_methods_text",
        "path": str(MANUSCRIPT_RUN_DIR / "notebook14_robustness_methods.md"),
        "description": "Methods text for robustness section.",
    },
    {
        "artifact_type": "manuscript",
        "name": "robustness_results_text",
        "path": str(MANUSCRIPT_RUN_DIR / "notebook14_robustness_results.md"),
        "description": "Results text for robustness section.",
    },
]

for _, row in figure_index.iterrows():
    output_rows.append({
        "artifact_type": "figure",
        "name": row["figure_id"],
        "path": row["path"],
        "description": row["caption"],
    })

output_index = pd.DataFrame(output_rows)

save_table(
    output_index,
    local_name="notebook14_output_index.csv",
    global_name=f"table_14_11_output_index_{RUN_ID}.csv",
)

print("Output index:")
print(output_index.to_string(index=False))

# ============================================================
# 16. Validation report and manifest
# ============================================================

print("\n" + "=" * 90)
print("Step 13: Saving validation report and manifest")
print("=" * 90)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "14_AURORA_transaction_cost_and_rebalance_sensitivity.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "purpose": (
        "Robustness analysis for transaction costs and rebalance frequency. "
        "Tests whether AURORA10-UAMV-B risk-control conclusions are sensitive "
        "to implementation assumptions."
    ),
    "input_paths": {
        "notebook13B_matrix": str(NOTEBOOK13B_MATRIX_PATH),
        "aurora_weights_path": str(aurora_weight_path) if aurora_weight_path else None,
        "roma_weights_path": str(roma_weight_path) if roma_weight_path else None,
        "etf_return_panel_path": str(ETF_RETURN_PANEL_PATH),
    },
    "primary_sensitivity_source": primary_sensitivity_source,
    "transaction_cost_bps_grid": TRANSACTION_COST_BPS_GRID,
    "rebalance_frequencies": list(REBALANCE_FREQUENCIES.keys()),
    "primary_method": PRIMARY_AURORA_POLICY,
    "primary_baseline": PRIMARY_ROMA_POLICY,
    "strict_test_period": {
        "n_days": int(len(source_aware_mat)),
        "start_date": str(source_aware_mat.index.min().date()),
        "end_date": str(source_aware_mat.index.max().date()),
    },
    "claim_stability_summary": claim_stability_summary.to_dict(orient="records"),
    "interpretation": (
        "The robustness analysis should be used to test fragility of the main claim, "
        "not to select a new best strategy on the strict-test period. The paper-safe "
        "claim remains downside-risk and risk-adjusted robustness, not total-return dominance."
    ),
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "returns": str(RETURN_RUN_DIR),
        "figures": str(PAPER_FIGURE_DIR),
        "reports": str(REPORT_RUN_DIR),
        "manuscript_assets": str(MANUSCRIPT_RUN_DIR),
    },
    "educational_note": (
        "This notebook is for reproducible financial machine-learning research only. "
        "It does not provide personalized financial advice or performance guarantees."
    ),
}

validation_report_path = REPORT_RUN_DIR / "NOTEBOOK14_validation_report.json"
validation_report_global_path = GLOBAL_REPORT_DIR / f"NOTEBOOK14_validation_report_{RUN_ID}.json"

write_json(validation_report_path, validation_report)
write_json(validation_report_global_path, validation_report)

manifest = make_file_manifest(RUN_ROOT)
manifest_path = REPORT_RUN_DIR / "NOTEBOOK14_file_manifest_SHA256.csv"
manifest_global_path = GLOBAL_REPORT_DIR / f"NOTEBOOK14_file_manifest_SHA256_{RUN_ID}.csv"

manifest.to_csv(manifest_path, index=False)
manifest.to_csv(manifest_global_path, index=False)

# ============================================================
# 17. Final summary
# ============================================================

print("\n" + "=" * 90)
print("NOTEBOOK 14 COMPLETE")
print("=" * 90)
print("Run ID                              :", RUN_ID)
print("Run root                            :", RUN_ROOT)
print("Primary sensitivity source          :", primary_sensitivity_source)
print("Original 13B performance table      :", TABLE_RUN_DIR / "notebook14_00_original_notebook13B_performance.csv")
print("Standardized weights table          :", TABLE_RUN_DIR / "notebook14_01_standardized_weights_long.csv")
print("True resimulation performance       :", TABLE_RUN_DIR / "notebook14_02_true_weight_resimulation_performance.csv")
print("Approx sensitivity performance      :", TABLE_RUN_DIR / "notebook14_03_approx_return_series_performance.csv")
print("Primary sensitivity performance     :", TABLE_RUN_DIR / "notebook14_04_primary_sensitivity_performance.csv")
print("Pairwise robustness table           :", TABLE_RUN_DIR / "notebook14_05_primary_sensitivity_pairwise_differences.csv")
print("Claim stability by scenario         :", TABLE_RUN_DIR / "notebook14_06_claim_stability_by_scenario.csv")
print("Claim stability summary             :", TABLE_RUN_DIR / "notebook14_07_claim_stability_summary.csv")
print("Paper robustness performance        :", TABLE_RUN_DIR / "notebook14_08_paper_robustness_performance_table.csv")
print("Paper robustness pairwise           :", TABLE_RUN_DIR / "notebook14_09_paper_robustness_pairwise_table.csv")
print("Figure index                        :", TABLE_RUN_DIR / "notebook14_10_figure_index.csv")
print("Paper figures                       :", PAPER_FIGURE_DIR)
print("Manuscript assets                   :", MANUSCRIPT_RUN_DIR)
print("Output index                        :", TABLE_RUN_DIR / "notebook14_output_index.csv")
print("Validation report                   :", validation_report_path)
print("Manifest                            :", manifest_path)
print("=" * 90)

print("\nClaim stability summary:")
print(claim_stability_summary.to_string(index=False))

print("\nPaper-safe robustness conclusion:")
print(
    "Notebook 14 tests whether the AURORA risk-control conclusion is sensitive to "
    "transaction-cost and rebalance-frequency assumptions. It should not be used to "
    "select a new best model. The preferred paper framing remains: AURORA10-UAMV-B "
    "is a downside-risk and risk-adjusted allocation layer, not a total-return-dominant "
    "trading strategy."
)

Mounted at /content/drive
Notebook 14: Transaction Cost and Rebalance Sensitivity
Timestamp UTC              : 2026-06-26T00:15:58Z
Run ID                     : 20260626_001558
Notebook 13B root          : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_AURORA_TWETF/source_aware_unified_paper_comparison/run_20260625_065916
Notebook 13B matrix        : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_AURORA_TWETF/source_aware_unified_paper_comparison/run_20260625_065916/returns/notebook13B_source_aware_strict_test_return_matrix.parquet
AURORA N10 root            : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/uncertainty_aware_mean_variance_allocation/run_20260624_100748
ROMA R2 root               : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_TWETF/aligned_allocation_backtest/run_20260625_025314
ETF return panel           : /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_return_panel.parquet
Run root                   : /content/drive/MyDrive/AURORA_TWETF/out